# Marginal Effects: Exercise Solutions

**Tutorial Series**: Discrete Choice Econometrics with PanelBox

**Notebook**: 04 - Marginal Effects (Solutions)

**Author**: PanelBox Contributors

**Date**: 2026-02-18

---

This notebook contains complete solutions for all exercises from the Marginal Effects tutorial.

## Setup

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import expit  # logistic CDF
from scipy.stats import norm

from panelbox.marginal_effects import compute_ame, compute_mem, compute_mer
from panelbox.models.discrete.binary import PooledLogit, PooledProbit

warnings.filterwarnings("ignore")
np.random.seed(42)
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

DATA_DIR = Path("..") / "data"
OUTPUT_DIR = Path("..") / "outputs"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Load labor participation data
data = pd.read_csv(DATA_DIR / "labor_participation.csv")

# Fit base Logit model (used across exercises)
formula_base = "lfp ~ age + educ + kids + married"
model_logit = PooledLogit(formula_base, data, "id", "year")
res_logit = model_logit.fit(cov_type="cluster")

print("Setup complete. Base model fitted.")
print(f"N obs: {res_logit.nobs}")
print("Coefficients:")
print(res_logit.params.round(6))

---

## Exercise 1: Manual AME Calculation (Easy)

**Task**: Replicate the AME calculation from scratch for the Probit model and verify against PanelBox.

In [ ]:
# Exercise 1 Solution

# Step 1: Estimate Probit model
model_probit = PooledProbit(formula_base, data, "id", "year")
res_probit = model_probit.fit(cov_type="cluster")

print("=== Probit Coefficients ===")
print(res_probit.params.round(6))

# Step 2: Extract design matrix and coefficients
if hasattr(model_probit, "exog_df"):
    X = model_probit.exog_df.values
    var_names = model_probit.exog_df.columns.tolist()
else:
    X = model_probit.exog
    var_names = model_probit.exog_names

beta = res_probit.params.values

print(f"\nDesign matrix shape: {X.shape}")
print(f"Variables: {var_names}")

# Step 3: Compute AME manually using the normal PDF formula
# For Probit: AME_k = (1/N) * sum_i [beta_k * phi(X_i' beta)]
# where phi is the standard normal PDF
#
# IMPORTANT: PanelBox auto-detects binary variables and uses
# discrete-difference instead of derivative. For a fair comparison,
# we replicate both approaches:
#   - Derivative for continuous variables (age, educ, kids)
#   - Discrete difference for binary variables (married)

xb = X @ beta  # Linear predictor for each observation
phi_xb = norm.pdf(xb)  # Standard normal PDF at X'beta


# Helper: check if a column is binary 0/1
def is_binary(x, tol=1e-10):
    unique_vals = np.unique(x)
    if len(unique_vals) == 2:
        return np.allclose(sorted(unique_vals), [0, 1], atol=tol)
    elif len(unique_vals) == 1:
        return unique_vals[0] in [0, 1]
    return False


manual_ame = {}
for j, var in enumerate(var_names):
    if var == "Intercept":
        # Skip: AME of the intercept is not economically meaningful.
        # PanelBox computes it via discrete-difference (since the intercept
        # column is all 1s, it is detected as binary), but we skip it.
        continue
    elif is_binary(X[:, j]):
        # Discrete difference: P(y=1|d=1, X) - P(y=1|d=0, X)
        X1 = X.copy()
        X1[:, j] = 1
        X0 = X.copy()
        X0[:, j] = 0
        prob1 = norm.cdf(X1 @ beta)
        prob0 = norm.cdf(X0 @ beta)
        manual_ame[var] = np.mean(prob1 - prob0)
    else:
        # Derivative: beta_k * phi(X'beta), averaged
        manual_ame[var] = np.mean(beta[j] * phi_xb)

manual_ame_series = pd.Series(manual_ame)

# Step 4: Compute AME using PanelBox
panelbox_ame = compute_ame(res_probit)

# Filter PanelBox results to exclude Intercept
pb_vars = [v for v in panelbox_ame.marginal_effects.index if v != "Intercept"]
pb_ame = panelbox_ame.marginal_effects[pb_vars]

# Step 5: Comparison table
print("\n=== AME Comparison: Manual vs PanelBox ===")
comparison = pd.DataFrame(
    {
        "Manual AME": manual_ame_series,
        "PanelBox AME": pb_ame,
        "Difference": manual_ame_series - pb_ame,
        "Match?": (manual_ame_series - pb_ame).abs() < 1e-6,
    }
)
print(comparison.round(8))

max_diff = (manual_ame_series - pb_ame).abs().max()
print(f"\nMax absolute difference: {max_diff:.2e}")
print(f"All match within 1e-6? {max_diff < 1e-6}")

# Step 6: Visualization — manual vs PanelBox bar comparison
fig, ax = plt.subplots(figsize=(10, 6))

x_pos = np.arange(len(pb_vars))
width = 0.35
ax.bar(
    x_pos - width / 2,
    manual_ame_series[pb_vars].values,
    width,
    label="Manual AME",
    color="#3498db",
    alpha=0.8,
    edgecolor="black",
)
ax.bar(
    x_pos + width / 2,
    pb_ame.values,
    width,
    label="PanelBox AME",
    color="#e74c3c",
    alpha=0.8,
    edgecolor="black",
)

ax.set_xticks(x_pos)
ax.set_xticklabels(pb_vars)
ax.set_ylabel("Average Marginal Effect")
ax.set_title("Exercise 1: Manual vs PanelBox AME (Probit)", fontweight="bold")
ax.legend()
ax.axhline(0, color="black", linewidth=0.8)
ax.grid(True, alpha=0.3, axis="y")

# Annotate differences
for i, var in enumerate(pb_vars):
    diff = comparison.loc[var, "Difference"]
    ymax = max(manual_ame_series[var], pb_ame[var])
    ymin = min(manual_ame_series[var], pb_ame[var])
    y_label = ymax + 0.002 if ymax >= 0 else ymin - 0.004
    ax.text(i, y_label, f"diff={diff:.1e}", ha="center", fontsize=8, style="italic")

plt.tight_layout()
plt.savefig(FIG_DIR / "04_manual_ame_validation.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n=== Probit AME Formula ===")
print("For Probit: AME_k = (1/N) sum_i [beta_k * phi(X_i' beta)]")
print("For Logit:  AME_k = (1/N) sum_i [beta_k * Lambda(X_i' beta)(1 - Lambda(X_i' beta))]")
print("The key difference is the PDF: normal phi (Probit) vs logistic (Logit).")
print("In practice, Logit and Probit AMEs are very similar.")
print("\nNote: For binary variables (married), PanelBox uses discrete-difference")
print("  P(y=1|d=1) - P(y=1|d=0) instead of the derivative formula.")
print("  Our manual computation matches this approach.")
print("\nFigure saved to outputs/figures/04_manual_ame_validation.png")

---

## Exercise 2: Dummy Variable Marginal Effect (Medium)

**Task**: Compare the derivative vs discrete difference methods for computing marginal effects of binary variables.

In [ ]:
# Exercise 2 Solution

# Step 1: Check if 'kids' is a dummy variable
print("=== Investigating 'kids' Variable ===")
print(f"Unique values: {sorted(data['kids'].unique())}")
print("Value counts:")
print(data["kids"].value_counts().sort_index())
print(
    f"\n'kids' is {'binary' if data['kids'].nunique() == 2 else 'NOT binary'} — "
    f"it has {data['kids'].nunique()} unique values."
)

# Step 2: Compute standard derivative AME for 'kids' (treating as continuous)
ame_deriv = compute_ame(res_logit)
print("\n=== Standard Derivative AME for 'kids' ===")
print(f"  AME (derivative): {ame_deriv.marginal_effects['kids']:.6f}")

# Step 3: Create binary indicator has_kids
data_ex2 = data.copy()
data_ex2["has_kids"] = (data_ex2["kids"] > 0).astype(int)

print("\n=== 'has_kids' Binary Variable ===")
print(f"has_kids = 0: {(data_ex2['has_kids'] == 0).sum()}")
print(f"has_kids = 1: {(data_ex2['has_kids'] == 1).sum()}")

# Step 4: Refit model with has_kids
formula_hk = "lfp ~ age + educ + has_kids + married"
model_hk = PooledLogit(formula_hk, data_ex2, "id", "year")
res_hk = model_hk.fit(cov_type="cluster")

print("\nModel with has_kids:")
print(res_hk.params.round(6))

# Step 5: Derivative AME for has_kids
X_hk = model_hk.exog
beta_hk = res_hk.params.values
xb_hk = X_hk @ beta_hk
prob_hk = expit(xb_hk)
logit_pdf = prob_hk * (1 - prob_hk)

hk_idx = res_hk.params.index.tolist().index("has_kids")
ame_hk_deriv = np.mean(beta_hk[hk_idx] * logit_pdf)

# Step 6: Discrete difference AME for has_kids
# P(y=1 | has_kids=1, X) - P(y=1 | has_kids=0, X)
X_0 = X_hk.copy()
X_1 = X_hk.copy()
X_0[:, hk_idx] = 0
X_1[:, hk_idx] = 1

prob_0 = expit(X_0 @ beta_hk)
prob_1 = expit(X_1 @ beta_hk)
ame_hk_discrete = np.mean(prob_1 - prob_0)

# PanelBox's AME (auto-detects binary variables)
ame_hk_panelbox = compute_ame(res_hk)

print("\n=== AME Comparison for 'has_kids' ===")
print(f"  Derivative method:         {ame_hk_deriv:.6f}")
print(f"  Discrete difference:       {ame_hk_discrete:.6f}")
print(f"  PanelBox compute_ame:      {ame_hk_panelbox.marginal_effects['has_kids']:.6f}")
print(f"  Difference (deriv-disc):   {ame_hk_deriv - ame_hk_discrete:.6f}")

# Step 7: Visualization — compare continuous kids AME vs binary has_kids
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Derivative vs discrete-difference for has_kids
methods = ["Derivative", "Discrete\nDifference", "PanelBox\n(auto)"]
values = [ame_hk_deriv, ame_hk_discrete, ame_hk_panelbox.marginal_effects["has_kids"]]
colors = ["#3498db", "#e74c3c", "#2ecc71"]
bars = axes[0].bar(methods, values, color=colors, alpha=0.8, edgecolor="black")
for bar, val in zip(bars, values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() - 0.005,
        f"{val:.4f}",
        ha="center",
        va="top",
        fontsize=10,
        fontweight="bold",
    )
axes[0].set_ylabel("Average Marginal Effect")
axes[0].set_title("has_kids: Derivative vs Discrete Difference", fontweight="bold")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].grid(True, alpha=0.3, axis="y")

# Right: Continuous kids AME vs binary has_kids AME
labels = ["Continuous kids\n(per-child effect)", "Binary has_kids\n(any vs none)"]
vals_compare = [ame_deriv.marginal_effects["kids"], ame_hk_discrete]
bars2 = axes[1].bar(
    labels, vals_compare, color=["#9b59b6", "#e74c3c"], alpha=0.8, edgecolor="black", width=0.5
)
for bar, val in zip(bars2, vals_compare):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() - 0.005,
        f"{val:.4f}",
        ha="center",
        va="top",
        fontsize=10,
        fontweight="bold",
    )
axes[1].set_ylabel("Average Marginal Effect")
axes[1].set_title("Continuous vs Binary: Effect of Children", fontweight="bold")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].grid(True, alpha=0.3, axis="y")

plt.suptitle("Exercise 2: Dummy Variable Marginal Effects", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_dummy_vs_continuous.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n=== Interpretation ===")
print("For binary (dummy) variables, the DISCRETE DIFFERENCE is preferred because:")
print("  1. It represents the actual change: P(y=1|d=1) - P(y=1|d=0)")
print("  2. It doesn't assume linearity at the point of evaluation")
print("  3. The derivative method approximates a continuous change, which is")
print("     conceptually wrong for a 0/1 variable.")
print(
    f"\nHere the difference is {abs(ame_hk_deriv - ame_hk_discrete):.4f} — "
    f"{'small' if abs(ame_hk_deriv - ame_hk_discrete) < 0.01 else 'notable'} "
    f"but methodologically important."
)
print("\nFigure saved to outputs/figures/04_dummy_vs_continuous.png")

---

## Exercise 3: Logit vs Probit AME Comparison (Medium)

**Task**: Compare Average Marginal Effects from Logit and Probit models to determine if model choice matters.

In [ ]:
# Exercise 3 Solution

# Step 1: Estimate identical specifications
formula_full = "lfp ~ age + I(age**2) + educ + kids + married + exper"

logit_ex3 = PooledLogit(formula_full, data, "id", "year")
res_logit_ex3 = logit_ex3.fit(cov_type="cluster")

probit_ex3 = PooledProbit(formula_full, data, "id", "year")
res_probit_ex3 = probit_ex3.fit(cov_type="cluster")

print("=== Model Log-Likelihoods ===")
print(f"Logit  log-L: {res_logit_ex3.llf:.4f}")
print(f"Probit log-L: {res_probit_ex3.llf:.4f}")

# Step 2: Compute AME for both
ame_logit = compute_ame(res_logit_ex3)
ame_probit = compute_ame(res_probit_ex3)

# Exclude Intercept for comparison
vars_compare = [v for v in ame_logit.marginal_effects.index if v != "Intercept"]

# Step 3: Comparison table
print("\n=== AME Comparison: Logit vs Probit ===")
comp_table = pd.DataFrame(
    {
        "Logit AME": ame_logit.marginal_effects[vars_compare],
        "Probit AME": ame_probit.marginal_effects[vars_compare],
        "Abs Diff": (
            ame_logit.marginal_effects[vars_compare] - ame_probit.marginal_effects[vars_compare]
        ).abs(),
    }
)
print(comp_table.round(6))

# Summary statistics
logit_vals = ame_logit.marginal_effects[vars_compare].values
probit_vals = ame_probit.marginal_effects[vars_compare].values
mad = np.mean(np.abs(logit_vals - probit_vals))
corr = np.corrcoef(logit_vals, probit_vals)[0, 1]

print(f"\nMean Absolute Difference (MAD): {mad:.6f}")
print(f"Correlation: {corr:.6f}")

# Step 4: Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Scatter plot
axes[0].scatter(logit_vals, probit_vals, s=100, color="#2c3e50", zorder=5)
for i, var in enumerate(vars_compare):
    axes[0].annotate(
        var, (logit_vals[i], probit_vals[i]), textcoords="offset points", xytext=(8, 5), fontsize=9
    )

lims = [min(min(logit_vals), min(probit_vals)) * 1.2, max(max(logit_vals), max(probit_vals)) * 1.2]
axes[0].plot(lims, lims, "r--", linewidth=1.5, label="45-degree line")
axes[0].set_xlabel("Logit AME")
axes[0].set_ylabel("Probit AME")
axes[0].set_title("Logit vs Probit AME")
axes[0].legend()
axes[0].text(
    0.05,
    0.9,
    f"Corr = {corr:.4f}\nMAD = {mad:.4f}",
    transform=axes[0].transAxes,
    fontsize=11,
    bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.5},
)
axes[0].grid(True, alpha=0.3)

# Right: Side-by-side bars
x_pos = np.arange(len(vars_compare))
width = 0.35
axes[1].bar(
    x_pos - width / 2,
    logit_vals,
    width,
    label="Logit AME",
    color="#3498db",
    alpha=0.8,
    edgecolor="black",
)
axes[1].bar(
    x_pos + width / 2,
    probit_vals,
    width,
    label="Probit AME",
    color="#e74c3c",
    alpha=0.8,
    edgecolor="black",
)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(vars_compare, rotation=30, ha="right")
axes[1].set_ylabel("Average Marginal Effect")
axes[1].set_title("AME by Variable")
axes[1].legend()
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].grid(True, alpha=0.3, axis="y")

plt.suptitle("Logit vs Probit: Average Marginal Effects Comparison", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_logit_probit_ame.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n=== Conclusion ===")
print(f"The AMEs from Logit and Probit are nearly identical (MAD = {mad:.4f}).")
print("Both models agree on:")
print("  - Sign (direction) of all effects")
print("  - Statistical significance")
print("  - Approximate magnitude")
print("\nPractical rule: Logit and Probit are interchangeable for AME purposes.")
print("Choose based on field convention, not expected AME differences.")
print("\nFigure saved to outputs/figures/04_logit_probit_ame.png")

---

## Exercise 4: Comprehensive Visualization (Hard)

**Task**: Create a 2x2 figure showing MEM vs AME vs MER, distribution of individual MEs, ME as function of age, and a forest plot with confidence intervals.

In [ ]:
# Exercise 4 Solution

# Prepare data: use base logit model
# Compute all three types of marginal effects
ame = compute_ame(res_logit)
mem = compute_mem(res_logit)
mer = compute_mer(res_logit, at={"age": 35, "educ": 12})

# Variables to plot (exclude Intercept)
plot_vars = [v for v in ame.marginal_effects.index if v != "Intercept"]

print("=== Marginal Effects Comparison ===")
me_compare = pd.DataFrame(
    {
        "AME": ame.marginal_effects[plot_vars],
        "MEM": mem.marginal_effects[plot_vars],
        "MER": mer.marginal_effects[plot_vars],
    }
)
print(me_compare.round(6))

# Compute individual-level MEs for educ
X = model_logit.exog
beta = res_logit.params.values
xb = X @ beta
prob = expit(xb)
logit_pdf = prob * (1 - prob)

educ_idx = res_logit.params.index.tolist().index("educ")
me_educ_individual = beta[educ_idx] * logit_pdf  # ME of educ for each observation

# Age values for panel (1,0)
ages = data["age"].values

# Create the 2x2 figure
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel (0,0): MEM vs AME vs MER bar chart
x_pos = np.arange(len(plot_vars))
width = 0.25
axes[0, 0].bar(
    x_pos - width,
    me_compare["AME"],
    width,
    label="AME",
    color="#3498db",
    alpha=0.8,
    edgecolor="black",
)
axes[0, 0].bar(
    x_pos, me_compare["MEM"], width, label="MEM", color="#e74c3c", alpha=0.8, edgecolor="black"
)
axes[0, 0].bar(
    x_pos + width,
    me_compare["MER"],
    width,
    label="MER (age=35, educ=12)",
    color="#2ecc71",
    alpha=0.8,
    edgecolor="black",
)
axes[0, 0].set_xticks(x_pos)
axes[0, 0].set_xticklabels(plot_vars)
axes[0, 0].set_ylabel("Marginal Effect")
axes[0, 0].set_title("AME vs MEM vs MER", fontweight="bold")
axes[0, 0].legend(fontsize=9)
axes[0, 0].axhline(0, color="black", linewidth=0.8)
axes[0, 0].grid(True, alpha=0.3, axis="y")

# Panel (0,1): Distribution of individual MEs for educ
axes[0, 1].hist(me_educ_individual, bins=50, color="#9b59b6", alpha=0.7, edgecolor="black")
me_mean = me_educ_individual.mean()
me_median = np.median(me_educ_individual)
axes[0, 1].axvline(me_mean, color="red", linestyle="--", linewidth=2, label=f"Mean = {me_mean:.4f}")
axes[0, 1].axvline(
    me_median, color="blue", linestyle="--", linewidth=2, label=f"Median = {me_median:.4f}"
)
axes[0, 1].set_xlabel("Marginal Effect of Education")
axes[0, 1].set_ylabel("Frequency")
axes[0, 1].set_title("Distribution of Individual MEs (educ)", fontweight="bold")
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# Panel (1,0): ME of educ as function of age
sort_idx = np.argsort(ages)
ages_sorted = ages[sort_idx]
me_educ_sorted = me_educ_individual[sort_idx]

# Scatter with low alpha + binned averages
axes[1, 0].scatter(ages_sorted, me_educ_sorted, alpha=0.1, s=5, color="gray")

age_bins = np.arange(ages.min(), ages.max() + 2, 2)
bin_centers = []
bin_means = []
for i in range(len(age_bins) - 1):
    mask = (ages >= age_bins[i]) & (ages < age_bins[i + 1])
    if mask.sum() > 0:
        bin_centers.append((age_bins[i] + age_bins[i + 1]) / 2)
        bin_means.append(me_educ_individual[mask].mean())

axes[1, 0].plot(
    bin_centers, bin_means, "o-", color="#e74c3c", linewidth=2, markersize=6, label="Binned average"
)
axes[1, 0].set_xlabel("Age")
axes[1, 0].set_ylabel("Marginal Effect of Education")
axes[1, 0].set_title("ME of Education by Age", fontweight="bold")
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Panel (1,1): Forest plot with AME +/- 95% CI
ci = ame.conf_int()
ame_vals = ame.marginal_effects[plot_vars].values
ci_lower = ci.loc[plot_vars, "lower"].values
ci_upper = ci.loc[plot_vars, "upper"].values

y_pos_forest = np.arange(len(plot_vars))
axes[1, 1].errorbar(
    ame_vals,
    y_pos_forest,
    xerr=[ame_vals - ci_lower, ci_upper - ame_vals],
    fmt="o",
    color="#2c3e50",
    markersize=10,
    capsize=6,
    linewidth=2,
)
axes[1, 1].axvline(0, color="red", linestyle="--", linewidth=1.5, alpha=0.7)
axes[1, 1].set_yticks(y_pos_forest)
axes[1, 1].set_yticklabels(plot_vars)
axes[1, 1].set_xlabel("Average Marginal Effect")
axes[1, 1].set_title("AME with 95% Confidence Intervals", fontweight="bold")

# Add significance stars
for i, var in enumerate(plot_vars):
    p = ame.pvalues[var]
    star = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
    axes[1, 1].text(ci_upper[i] + 0.001, i, star, va="center", fontsize=12, color="red")

axes[1, 1].grid(True, alpha=0.3, axis="x")

plt.suptitle(
    "Comprehensive Marginal Effects Analysis \u2014 Labor Force Participation",
    fontweight="bold",
    fontsize=14,
    y=1.01,
)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_comprehensive_me.png", dpi=150, bbox_inches="tight")
plt.show()

# Print summary statistics
print("\n=== AME Summary with Standard Errors ===")
print(ame.summary())

print("\n=== Key Observations ===")
print("1. AME, MEM, and MER are similar but not identical.")
print("   MER depends on the representative point chosen.")
print("2. Individual MEs for 'educ' show substantial heterogeneity.")
print(f"   Range: [{me_educ_individual.min():.4f}, {me_educ_individual.max():.4f}]")
print("3. The ME of education varies with age due to the nonlinear model.")
print("4. All AMEs are statistically significant (CIs exclude 0).")
print("\nFigure saved to outputs/figures/04_comprehensive_me.png")

---

## Key Takeaways from Exercises

1. **Exercise 1**: The AME formula for Probit uses the normal PDF \u03c6(X'\u03b2) while Logit uses \u039b(X'\u03b2)(1\u2212\u039b(X'\u03b2)). Manual computation matches PanelBox when we correctly use discrete-difference for binary variables (like `married`) instead of the derivative formula.

2. **Exercise 2**: For binary (dummy) variables, the discrete difference method [P(y=1|d=1) \u2212 P(y=1|d=0)] is conceptually correct, while the derivative method approximates a continuous change that doesn't make sense for a 0/1 variable.

3. **Exercise 3**: Logit and Probit produce nearly identical AMEs despite different coefficient scales. Model choice does not meaningfully affect marginal effects \u2014 choose based on convention.

4. **Exercise 4**: Marginal effects in nonlinear models are heterogeneous across observations. The AME summarizes the average, but the full distribution reveals important variation. MEM and MER provide effects at specific covariate values, useful for policy analysis.

---

**End of Solutions**